# Train on the Global Wheat Head Dataset 2021

**Question this notebook answers:** *does adding training images from more farms and countries improve accuracy on farms the model has never seen?*

In the earlier notebook (`train.ipynb`) we found that a bigger model, higher resolution, longer training and extra augmentation did **not** improve the unseen-farm score. The evidence pointed at the **data** (only five training farms) as the limit. This notebook tests that hypothesis directly.

## Experiment design: change one thing

| | YOLO11s baseline | This notebook |
|---|---|---|
| Model | YOLO11s | YOLO11s (same) |
| Image size / batch / epochs | 1024 / 4 / 60 | 1024 / 4 / 60 (same) |
| Augmentation, seed, optimizer | defaults + the baseline settings | identical |
| **Training images** | 2,228 images, 5 farm groups | **2,228 + new images from new farms** |
| `val` and `test` images | unchanged | unchanged |

Because *only the training data differs*, any change in the score can be credited to the data.

## The dataset splits

| Split | What it is | How it is used |
|---|---|---|
| `train` | Old training images **plus** new farms from GWHD 2021 | Learning |
| `val` | The same 247 images as before | Early stopping and picking the best checkpoint |
| `test` | `usask_1` + `ethz_1` (947 images), same as before | Headline unseen-farm score. Scored once. |
| `test2` | The official GWHD 2021 *competition-test* farms (new, never trained on) | A second, independent unseen-farm score |

## The pipeline in this notebook

```
download 10 GB archive ──> read labels ──> fingerprint every image ──> find images we already have
        │                                                                        │
        └── skip duplicates + test farms ──> convert PNG→JPEG + labels ──> verify ──> train ──> evaluate on val / test / test2
```

## Rules we keep

1. **Never train on `test` or `test2`.** The 2021 archive contains *our own test images* under new file names, so the notebook detects and removes them. Excluding by farm name alone would not be safe (see section 4).
2. Choose between experiments on `val`. Look at `test` and `test2` only when reporting.

> **Licence note:** the dataset is public research data, but check the dataset page's licence and citation request before sharing a trained model or publishing results.

## 1. Setup: imports, paths and settings

**What it does:** imports the libraries, finds the repository root, and defines every path and tunable setting in one place.

**How it works:**
- `find_repo_root` walks up from the notebook's folder until it finds a directory containing both `data/` and `notebooks/`, so the notebook works no matter where it is launched from.
- Paths are built from that root, so nothing is hard-coded to one machine.
- The first block of constants are *experiment switches*; the second block are *cleaning rules* copied from `train.ipynb` so old and new data are treated identically.
- The final loop is a **precondition check**: the old `train/val/test` folders must already exist (created by `train.ipynb`), because we reuse the same `val` and `test` images.

**Settings worth knowing:**
- `HOLDOUT_OFFICIAL_TEST`: keeps the official 2021 test farms out of training so they can serve as `test2`. Set it to `False` to train on them too (you then lose `test2`).
- `DUP_THRESHOLD`: how similar two thumbnails must be to count as the same photo (explained in section 5).
- `RUN_TRAINING`: set automatically from the environment variable `GWD_SKIP_TRAIN`. Setting it to `1` skips training, which is useful for testing the data and evaluation cells.
- `MODEL` (section 11): which YOLO11 size to train, taken from the environment variable `GWD_MODEL` (`yolo11s` by default, or `yolo11m`). Each size writes to its own run folder, so both can be trained and compared.

In [ ]:
import io, json, os, shutil, subprocess, sys, threading, time, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image


def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project repository.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
OLD = REPO_ROOT / "data" / "yolo"                  # existing dataset: train / val / test
OLD_JPG = REPO_ROOT / "data" / "raw" / "train"     # all 3,422 original Kaggle images
RAW21 = REPO_ROOT / "data" / "gwhd2021"            # the downloaded 2021 archive
NEW = REPO_ROOT / "data" / "yolo21"                # generated YOLO files for the new images
RUNS_DIR = REPO_ROOT / "runs" / "detect"
ZIP_PATH = RAW21 / "global-wheat-head-dataset-2021.zip"
KAGGLE_DATASET = "vbookshelf/global-wheat-head-dataset-2021"

# --- experiment switches
HOLDOUT_OFFICIAL_TEST = True   # official 2021 test farms stay out of training and become 'test2'
DUP_THRESHOLD = 3.0            # thumbnail distance below this = "same photo we already have"
THUMB = 16                     # thumbnail side in pixels
JPEG_QUALITY = 95

# --- cleaning rules (identical to train.ipynb)
MAX_BOX_AREA = 200_000         # bigger than this is an annotation error
MIN_BOX_SIDE = 2               # pixels

RUN_TRAINING = os.environ.get("GWD_SKIP_TRAIN") != "1"

for split in ("train", "val", "test"):
    assert (OLD / "images" / split).is_dir(), f"missing {OLD / 'images' / split} - run train.ipynb data preparation first"
NEW.mkdir(parents=True, exist_ok=True)
print("repo root     ", REPO_ROOT)
print("old train/val/test images:", {s: len(list((OLD / 'images' / s).glob('*.jpg'))) for s in ('train', 'val', 'test')})

## 2. Download the dataset from Kaggle

**What it does:** downloads the 2021 dataset (about **10 GB** because the images are lossless PNGs) into `data/gwhd2021/`, unless it is already there.

**How it works:**
- The Kaggle command-line tool does the download. It needs an API token, which the function `kaggle_env` looks for in the environment variable `KAGGLE_API_TOKEN`, or else in a git-ignored `.env` file at the repository root.
- The token is placed into the *child process's* environment only. It is never printed, logged or written anywhere else.
- If the archive already exists and is larger than 10 GB, the download is skipped, so re-running the notebook is cheap.

**When to expect trouble:** a `401` error means the token is wrong or expired; a `403` can mean the dataset's terms were not accepted on the Kaggle website. Downloads can take about 10 minutes.

**Never commit the token.** `.env` and `data/gwhd2021/` are listed in `.gitignore`.

In [ ]:
def kaggle_env():
    """Environment for the Kaggle CLI, with the API token taken from the environment or .env (never printed)."""
    env = os.environ.copy()
    if not env.get("KAGGLE_API_TOKEN"):
        dotenv = REPO_ROOT / ".env"
        if dotenv.exists():
            for line in dotenv.read_text().splitlines():
                if line.startswith("KAGGLE_API_TOKEN="):
                    env["KAGGLE_API_TOKEN"] = line.split("=", 1)[1].strip()
    return env


if ZIP_PATH.exists() and ZIP_PATH.stat().st_size > 10_000_000_000:
    print(f"archive already downloaded ({ZIP_PATH.stat().st_size / 1e9:.1f} GB) - skipping")
else:
    RAW21.mkdir(parents=True, exist_ok=True)
    kaggle_exe = Path(sys.executable).parent / "kaggle.exe"
    subprocess.run([str(kaggle_exe if kaggle_exe.exists() else "kaggle"), "datasets", "download",
                    "-d", KAGGLE_DATASET, "-p", str(RAW21)], check=True, env=kaggle_env())
    print("downloaded", ZIP_PATH)

## 3. Read the labels straight from the archive

**What it does:** reads the three label CSV files (`competition_train/val/test.csv`) out of the zip *without extracting the 10 GB of images*, and converts each box string into numbers.

**The label format is different from the original Kaggle data**, which is a classic trap:

| | Original Kaggle `train.csv` | GWHD 2021 CSV |
|---|---|---|
| Box layout | `[x_min, y_min, width, height]` | `x_min y_min x_max y_max` (two **corners**) |
| Separator | one string per box | boxes joined by `;` |
| Empty image | no rows | the text `no_box` |

Reading a corner format as "corner + size" would silently produce wrong boxes. The parser below follows the 2021 format, and section 7 later proves it with a round-trip check.

**How it works:** `parse_boxes` splits the string on `;` to get one box each, then on spaces to get four numbers. `no_box` becomes an empty array. The `official_split` column remembers which CSV (train, val or test) a row came from.

**Two data-quality problems handled here:**
- Three image names appear in two rows with **different boxes and domains**, so their labels are ambiguous. We drop those images.
- Three PNG files in the archive have **no label row**. They are unusable and are ignored.

Together that is under 0.1% of the data.

In [ ]:
zf = zipfile.ZipFile(ZIP_PATH)

labels = pd.concat(
    [pd.read_csv(io.BytesIO(zf.read(f"gwhd_2021/competition_{s}.csv"))).assign(official_split=s)
     for s in ("train", "val", "test")], ignore_index=True)


def parse_boxes(s):
    """'x1 y1 x2 y2;x1 y1 x2 y2' -> (N, 4) float array of corner boxes; 'no_box' -> (0, 4)."""
    if not isinstance(s, str) or s == "no_box":
        return np.zeros((0, 4), dtype=np.float32)
    return np.array([[float(v) for v in box.split()] for box in s.split(";")], dtype=np.float32)


labels["n_boxes"] = labels["BoxesString"].map(lambda s: len(parse_boxes(s)))
print(f"{len(labels):,} label rows | {labels.n_boxes.sum():,} boxes | {(labels.n_boxes == 0).sum()} images with no boxes")

png_names = sorted({Path(n).name for n in zf.namelist() if n.endswith(".png")})

unlabelled = sorted(set(png_names) - set(labels.image_name))              # PNGs with no label row at all
ambiguous = labels.image_name[labels.image_name.duplicated(keep=False)].unique()
labels = labels[~labels.image_name.isin(ambiguous)].set_index("image_name")
png_names = [n for n in png_names if n in labels.index]
print(f"dropped {len(ambiguous)} images with ambiguous labels and {len(unlabelled)} PNGs with no label row")
print(f"usable labelled images: {len(png_names):,}")
labels.head(3)

## 4. What is in the dataset? Domains, countries and the renaming trap

**What it does:** summarizes the dataset per **domain** (one farm / acquisition session) and joins in the country and growth stage from the archive's `metadata_dataset.csv`.

**How it works:** `groupby("domain")` counts images and boxes; the metadata file (semicolon-separated) supplies country and development stage. Names differ only in capitalization (`ETHZ_1` vs `Ethz_1`), so the join key is lower-cased.

**Why look at this:** it shows why the new data can help. The original data came from a few European sites plus Canada. The 2021 version adds Japan, Norway, Belgium, Australia, the US, Mexico, Sudan and China, with different crops stages and cameras.

**The renaming trap (important):** the 2021 version **re-grouped the original farms under new names with different sizes.** For example, the Kaggle `arvalis_1` (1,055 images) is now spread across `Arvalis_2`, `Arvalis_5`, `Arvalis_6`, and others. So *matching farm names between the old and new data would not tell us which images we already have.* If we only excluded `ETHZ_1` and `Usask_1` by name and trusted the other names, we could not be sure no test image leaks into training. Section 5 solves this by comparing the **pictures themselves**.

In [ ]:
meta = pd.read_csv(io.BytesIO(zf.read("gwhd_2021/metadata_dataset.csv")), sep=";")
meta["key"] = meta["name"].str.lower()
labels["key"] = labels["domain"].str.lower()

domains = (labels.groupby("domain")
           .agg(images=("n_boxes", "size"), boxes=("n_boxes", "sum"), official_split=("official_split", "first"),
                key=("key", "first"))
           .reset_index().merge(meta[["key", "country", "development_stage"]], on="key", how="left")
           .drop(columns="key"))
domains["boxes_per_image"] = (domains.boxes / domains.images).round(1)
print(f"{len(domains)} domains in {domains.country.nunique()} countries")
display(domains.sort_values(["official_split", "domain"]).reset_index(drop=True))
display(domains.groupby("country").agg(domains=("domain", "size"), images=("images", "sum")).sort_values("images", ascending=False))

## 5. Find the images we already have (leakage defence)

The 2021 archive **contains all 3,422 original images**, saved as PNG under new hash-like names (`4563856c….png` instead of `b6ab77fd7.jpg`). If we trained on all of it we would train on our own `val` and `test` images, which would silently destroy the honest evaluation. So we must identify, picture by picture, which new files are old images.

### 5a. Fingerprint every image

**What it does:** shrinks every image (new PNGs and old JPEGs) to a **16 × 16 grayscale thumbnail**, which is a 256-number fingerprint.

**How it works:**
- `thumbnail()` converts to grayscale and downsizes with a box filter (averaging blocks of pixels), which ignores small differences such as JPEG compression versus PNG.
- Images are read by a **thread pool** (12 threads). PNG decoding and resizing release Python's global lock, so threads give a real speed-up without needing separate processes.
- Each thread keeps its own `ZipFile` handle (`threading.local`) so threads do not fight over one file pointer.
- Results are cached in `data/yolo21/fingerprints.npz`, so the next run takes a second instead of about 20 s.

**Why thumbnails work:** two copies of the same photo give almost identical thumbnails, while two different wheat photos differ a lot. Real photos of wheat are different enough at pixel level that accidental matches are essentially impossible, as the next step shows.

In [ ]:
FP_CACHE = NEW / "fingerprints.npz"
old_paths = sorted(OLD_JPG.glob("*.jpg"))


def thumbnail(im):
    return np.asarray(im.convert("L").resize((THUMB, THUMB), Image.BOX), dtype=np.uint8).ravel()


_local = threading.local()


def new_fingerprint(name):
    if not hasattr(_local, "zf"):                     # one ZipFile handle per thread
        _local.zf = zipfile.ZipFile(ZIP_PATH)
    return thumbnail(Image.open(io.BytesIO(_local.zf.read(f"gwhd_2021/images/{name}"))))


def old_fingerprint(path):
    return thumbnail(Image.open(path))


cache_ok = False
if FP_CACHE.exists():
    c = np.load(FP_CACHE)
    cache_ok = list(c["new_names"]) == png_names and list(c["old_names"]) == [p.name for p in old_paths]
if cache_ok:
    new_fp, old_fp = c["new"], c["old"]
    print("fingerprints loaded from cache")
else:
    t0 = time.time()
    with ThreadPoolExecutor(12) as pool:
        new_fp = np.stack(list(pool.map(new_fingerprint, png_names)))
        old_fp = np.stack(list(pool.map(old_fingerprint, old_paths)))
    np.savez(FP_CACHE, new_names=np.array(png_names), new=new_fp, old_names=np.array([p.name for p in old_paths]), old=old_fp)
    print(f"fingerprinted {len(new_fp):,} new and {len(old_fp):,} old images in {time.time() - t0:.0f}s")
print(new_fp.shape, old_fp.shape)

### 5b. Distance to the nearest image we already own

**What it does:** for every new image, finds the *most similar* old image and measures how different they are.

**How it works:** `cdist(..., "cityblock")` computes the **L1 distance** (sum of absolute pixel differences) between every pair of fingerprints at once; dividing by 256 turns it into a *mean absolute difference per pixel*, on a 0 to 255 scale. `.min(axis=1)` keeps the closest old image for each new one.

**How to read the histogram:** if some new images are copies of old ones, the distances form **two separate clusters**: a spike near 0 (copies) and a big hump far away (genuinely new photos). A clean gap between them means any threshold in the gap gives the same answer, so the result does not depend on a delicate guess. The cell prints how many images fall in the "ambiguous" band between the clusters; the expected answer is **0**.

In [ ]:
from scipy.spatial.distance import cdist

D = cdist(new_fp.astype(np.float32), old_fp.astype(np.float32), metric="cityblock") / new_fp.shape[1]
nearest = D.argmin(axis=1)
nearest_dist = D.min(axis=1)

fp = pd.DataFrame({"name": png_names, "dist": nearest_dist, "old_match": [old_paths[i].name for i in nearest]})
fp["domain"] = fp.name.map(labels["domain"])
fp["official_split"] = fp.name.map(labels["official_split"])
fp["is_dup"] = fp.dist < DUP_THRESHOLD

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(np.clip(nearest_dist, 0.01, 100), bins=np.logspace(-2, 2, 80), color="#4a7c59")
ax.set_xscale("log"); ax.axvline(DUP_THRESHOLD, color="crimson", ls="--", label=f"threshold = {DUP_THRESHOLD}")
ax.set_xlabel("mean pixel difference to the nearest old image (log scale)"); ax.set_ylabel("new images"); ax.legend()
plt.tight_layout(); plt.show()

band = ((nearest_dist >= 0.5) & (nearest_dist < 8.0)).sum()
print(f"near-duplicates of images we already have: {fp.is_dup.sum():,}  |  genuinely new: {(~fp.is_dup).sum():,}")
print(f"images in the ambiguous band 0.5-8.0: {band}  (0 means the two groups are cleanly separated)")

### 5c. Prove the method is correct

**What it does:** two independent checks that the matching found exactly the right images.

1. **Test farms:** the archive's `ETHZ_1` + `Usask_1` images must match *our* `test` images one-to-one. We know our test set is exactly those two farms (947 images), so if the method works the match count is 947 with no repeats.
2. **Every old image is found:** all 3,422 original images should each be matched by some new file. If a few were missing, it would mean our fingerprints were too strict.

**Why this matters:** instead of trusting a threshold, we verify against ground truth we already know. The cell raises an error (`assert`) if either check fails, so a bad match can never reach training.

In [ ]:
old_test = {p.name for p in (OLD / "images" / "test").glob("*.jpg")}
old_val = {p.name for p in (OLD / "images" / "val").glob("*.jpg")}

farms = fp[fp.domain.isin(["ETHZ_1", "Usask_1"])]
print(f"ETHZ_1 + Usask_1 images in the archive: {len(farms)}")
print(f"  matched to our TEST images: {farms.old_match.isin(old_test).sum()}  (unique: {farms.old_match.nunique()})")
assert len(farms) == len(old_test) == farms.old_match.isin(old_test).sum() == farms.old_match.nunique(), "test-set mapping failed"

dups = fp[fp.is_dup]
print(f"duplicates matching our VAL images: {dups.old_match.isin(old_val).sum()} | distinct old images matched: {dups.old_match.nunique()} of {len(old_paths)}")
assert dups.old_match.nunique() == len(old_paths), "some old images were not found in the archive"
print("OK: the 2021 archive contains all our old images, and our test set maps exactly onto ETHZ_1 + Usask_1")

## 6. Decide what is new, what is training, what is `test2`

**What it does:** splits the genuinely new images (those *not* near-duplicates of anything we have) into training images and the second test set.

**How it works:**
- `cand` = images with no old twin. By construction none of them are in our `train`, `val` or `test`.
- If `HOLDOUT_OFFICIAL_TEST` is true, new images from the official **competition-test** CSV become `test2`. These are 18 farm groups from the US, Mexico, Japan, Australia and others.
- Everything else new (from the official train and val CSVs) is added to our training set.

**Why hold out `test2`:** a single test set of two farms can be lucky or unlucky. A second set of **different farms on different continents**, never trained on, makes the final conclusion much more trustworthy, and we can score the two earlier models on it for free (they were trained before this data existed, so there is no leakage for them either).

**The trade-off:** `test2` is 1,380 images we do *not* train on, so training has less new data than it could. If the experiment helps, a final model trained on everything except our two test farms is the natural next step.

In [ ]:
cand = fp[~fp.is_dup].copy()
if HOLDOUT_OFFICIAL_TEST:
    test2 = cand[cand.official_split == "test"]
else:
    test2 = cand.iloc[0:0]
add_train = cand.drop(test2.index)

for name, part in (("new TRAINING images", add_train), ("test2 (held out)", test2)):
    print(f"{name}: {len(part):,} images from {part.domain.nunique()} domains")

summary = (pd.concat([add_train.assign(role="train"), test2.assign(role="test2")])
           .merge(domains[["domain", "country"]], on="domain")
           .groupby(["role", "country"]).agg(images=("name", "size"), domains=("domain", "nunique")))
display(summary)
print(f"\ntraining set will be {len(list((OLD / 'images' / 'train').glob('*.jpg'))):,} old + {len(add_train):,} new = "
      f"{len(list((OLD / 'images' / 'train').glob('*.jpg'))) + len(add_train):,} images")

## 7. Convert the new images to JPEG and write YOLO labels

**What it does:** for every selected image it (1) decodes the PNG from the archive, (2) saves it as a high-quality JPEG, and (3) writes a YOLO label file next to it. The result lives under `data/yolo21/images|labels/{train,test2}`.

**How it works:**
1. *Corner to YOLO conversion.* A box `(x1, y1, x2, y2)` in pixels becomes `class xc yc w h`, all divided by the image size so that values lie between 0 and 1:
   `xc = (x1+x2)/2/W`, `yc = (y1+y2)/2/H`, `w = (x2-x1)/W`, `h = (y2-y1)/H`.
2. *Same cleaning as before.* Boxes are clipped to the image, and those with a side under 2 px or an area over 200,000 px² are dropped, exactly like `train.ipynb`.
3. *Why JPEG:* the original data was JPEG, loading a 1024² PNG is slower than a JPEG, and 10 GB of PNG would slow every epoch. Quality 95 is visually lossless. It also keeps old and new images in the **same format**, so the model cannot learn "PNG vs JPEG" as a shortcut.
4. *Empty images (`no_box`)* get an empty `.txt` file. That is valid YOLO: it means "background image, nothing to detect".
5. The work runs on a thread pool, and the cell is **idempotent**: files that already exist are skipped, so you can re-run it safely.

**Where YOLO finds labels:** Ultralytics replaces the folder name `images` with `labels` in each image path, so the matching label file must sit in the parallel `labels/` folder with the same file name. Our layout follows that rule.

In [ ]:
box_strings = labels["BoxesString"].to_dict()
for split in ("train", "test2"):
    (NEW / "images" / split).mkdir(parents=True, exist_ok=True)
    (NEW / "labels" / split).mkdir(parents=True, exist_ok=True)


def convert_one(job):
    name, split = job
    stem = Path(name).stem
    img_out = NEW / "images" / split / f"{stem}.jpg"
    lbl_out = NEW / "labels" / split / f"{stem}.txt"
    if img_out.exists() and lbl_out.exists():
        return dict(name=name, split=split, status="skipped", kept=len(lbl_out.read_text().split("\n")) if lbl_out.stat().st_size else 0, dropped=0)
    if not hasattr(_local, "zf"):
        _local.zf = zipfile.ZipFile(ZIP_PATH)
    im = Image.open(io.BytesIO(_local.zf.read(f"gwhd_2021/images/{name}"))).convert("RGB")
    W, H = im.size
    b = parse_boxes(box_strings[name])
    x1, y1 = np.clip(b[:, 0], 0, W), np.clip(b[:, 1], 0, H)           # clip to the image
    x2, y2 = np.clip(b[:, 2], 0, W), np.clip(b[:, 3], 0, H)
    w, h = x2 - x1, y2 - y1
    keep = (w >= MIN_BOX_SIDE) & (h >= MIN_BOX_SIDE) & (w * h <= MAX_BOX_AREA)
    lines = [f"0 {(a + c) / 2 / W:.6f} {(d + e) / 2 / H:.6f} {ww / W:.6f} {hh / H:.6f}"
             for a, d, c, e, ww, hh in zip(x1[keep], y1[keep], x2[keep], y2[keep], w[keep], h[keep])]
    im.save(img_out, "JPEG", quality=JPEG_QUALITY)
    lbl_out.write_text("\n".join(lines))
    return dict(name=name, split=split, status="converted", kept=len(lines), dropped=int((~keep).sum()))


jobs = [(n, "train") for n in add_train.name] + [(n, "test2") for n in test2.name]
t0 = time.time()
with ThreadPoolExecutor(12) as pool:
    conv = pd.DataFrame(list(pool.map(convert_one, jobs)))
print(f"{conv.status.value_counts().to_dict()} in {time.time() - t0:.0f}s")
print(f"boxes kept {conv.kept.sum():,} | dropped by cleaning rules {conv.dropped.sum():,}")
conv.groupby("split").agg(images=("name", "size"), boxes=("kept", "sum"))

## 8. Verify the conversion before spending hours training

Silent label bugs do not raise errors. They just make a model learn nonsense. This section is cheap insurance with three independent checks.

### 8a. Range check
**What it does:** reads *every* new label file and confirms the class is 0 and all numbers lie in [0, 1], with each box inside the image. **How:** a loop over the files; `assert bad == 0` stops the notebook if even one label is invalid.

In [ ]:
bad = total = empty = 0
for f in (NEW / "labels").rglob("*.txt"):
    text = f.read_text().strip()
    if not text:
        empty += 1
        continue
    for line in text.splitlines():
        p = line.split(); total += 1
        cls = int(p[0]); xc, yc, w, h = map(float, p[1:])
        if cls != 0 or not all(0 <= v <= 1 for v in (xc, yc, w, h)):
            bad += 1
        if xc - w / 2 < -1e-6 or xc + w / 2 > 1 + 1e-6 or yc - h / 2 < -1e-6 or yc + h / 2 > 1 + 1e-6:
            bad += 1
print(f"{total:,} labels checked in {len(list((NEW / 'labels').rglob('*.txt'))):,} files ({empty} background-only), {bad} invalid")
assert bad == 0, "conversion bug - do not train until this is 0"

### 8b. Round trip: convert one box back to pixels

**What it does:** takes one image, converts its first YOLO label *back* to pixel corners, and compares with the original `x1 y1 x2 y2` text in the CSV.

**Why:** this is the check that catches the corner-vs-size mix-up described in section 3. If the format were read wrongly, the "recovered" numbers would not match the "original" ones. Small differences (a pixel or less) are expected, since labels are stored with 6 decimals and some boxes are clipped.

In [ ]:
name = add_train.name.iloc[0]
orig = parse_boxes(box_strings[name])[0]
lbl = (NEW / "labels" / "train" / f"{Path(name).stem}.txt").read_text().splitlines()[0].split()
xc, yc, w, h = map(float, lbl[1:])
rec = np.array([(xc - w / 2) * 1024, (yc - h / 2) * 1024, (xc + w / 2) * 1024, (yc + h / 2) * 1024])
print("original  x1 y1 x2 y2 :", orig.round(1))
print("recovered x1 y1 x2 y2 :", rec.round(1))
assert np.abs(orig - rec).max() < 1.5, "round trip does not match"

### 8c. Look at the pictures

**What it does:** draws the converted boxes on eight images from eight different farms (some training, some `test2`), reading the boxes back **from the label files on disk**, not from memory.

**Why:** numbers can pass every check while still being drawn on the wrong thing. If boxes sit on wheat heads in every panel, the whole chain (PNG decode, JPEG save, label conversion, file layout) is proven end to end. If they are shifted, flipped or tiny, stop here.

In [ ]:
import matplotlib.patches as patches

rng = np.random.default_rng(0)
shown = pd.concat([add_train.assign(role="train"), test2.assign(role="test2")])
picks = shown.groupby("domain").sample(1, random_state=0).sample(8, random_state=1)

fig, axes = plt.subplots(2, 4, figsize=(18, 9.2))
for ax, r in zip(axes.ravel(), picks.itertuples()):
    stem = Path(r.name).stem
    ax.imshow(Image.open(NEW / "images" / r.role / f"{stem}.jpg"))
    txt = (NEW / "labels" / r.role / f"{stem}.txt").read_text().split("\n")
    n = 0
    for line in filter(None, txt):
        _, xc, yc, w, h = map(float, line.split())
        ax.add_patch(patches.Rectangle(((xc - w / 2) * 1024, (yc - h / 2) * 1024), w * 1024, h * 1024,
                                       fill=False, edgecolor="#00ff41", lw=1)); n += 1
    ax.set_title(f"{r.domain} ({r.role}) - {n} heads", fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

## 9. How different is the new data?

**What it does:** compares the old training images with the new ones on three properties: heads per image, box size in pixels, and background-only images.

**How it works:** it reads the YOLO label files of each group, counts boxes per image, and converts box width and height back to pixels (`sqrt(w*h) * 1024`, the "box side").

**Why it matters:** if the new farms looked exactly like the old ones, we should not expect an improvement. Large differences in head density or head size mean the model is being shown *new situations*, which is the point of the experiment. Detectors especially struggle when head size at test time falls outside the range seen in training.

In [ ]:
def label_stats(label_dir):
    rows = []
    for f in Path(label_dir).glob("*.txt"):
        t = f.read_text().strip()
        arr = np.array([l.split() for l in t.splitlines()], dtype=float) if t else np.zeros((0, 5))
        rows.append((f.stem, len(arr), np.sqrt(arr[:, 3] * arr[:, 4]) * 1024 if len(arr) else np.array([])))
    return rows


def describe(rows, label):
    counts = np.array([r[1] for r in rows]); sides = np.concatenate([r[2] for r in rows])
    return dict(group=label, images=len(rows), boxes=int(counts.sum()), heads_per_image=round(counts.mean(), 1),
                median_box_px=round(float(np.median(sides)), 0), p5_box_px=round(float(np.quantile(sides, .05)), 0),
                p95_box_px=round(float(np.quantile(sides, .95)), 0), background_only=int((counts == 0).sum()))


stats = pd.DataFrame([describe(label_stats(OLD / "labels" / "train"), "old train"),
                      describe(label_stats(NEW / "labels" / "train"), "new train"),
                      describe(label_stats(OLD / "labels" / "test"), "test (usask_1+ethz_1)"),
                      describe(label_stats(NEW / "labels" / "test2"), "test2")]).set_index("group")
stats

## 10. Build the combined training set and the dataset files

**What it does:** writes a **text file listing every training image** (old and new) and two YAML dataset descriptions that tell Ultralytics where the data is.

**How it works:**
- `train_gwhd21.txt` holds one absolute image path per line: the 2,228 old training images followed by the new ones. Ultralytics accepts a text file as a training source, so **no images are copied or duplicated** on disk. The old images keep their labels in `data/yolo/labels/train`, the new ones in `data/yolo21/labels/train`.
- `wheat21.yaml` points `train` at that list, and `val` / `test` at the **same old folders as before**. This keeps `val` and `test` numbers directly comparable with the earlier runs.
- `wheat21_test2.yaml` is only for scoring: its `val` is the `test2` folder, so any model can be evaluated on `test2` with the usual `val()` call.

**Why a list file and not a merged folder:** copying thousands of images would waste disk and risk accidentally mixing a `test` image into `train`. A list makes the exact membership of the training set explicit and auditable.

**Note on balanced sampling:** unlike the `yolo11m_combo` run, we do **not** repeat small farms here. The goal is to isolate the effect of the data, so everything else stays as in the baseline.

In [ ]:
train_list = sorted((OLD / "images" / "train").glob("*.jpg")) + sorted((NEW / "images" / "train").glob("*.jpg"))
(NEW / "train_gwhd21.txt").write_text("\n".join(str(p) for p in train_list) + "\n")

(NEW / "wheat21.yaml").write_text(
    f"path: {NEW}\ntrain: train_gwhd21.txt\nval: {OLD / 'images' / 'val'}\ntest: {OLD / 'images' / 'test'}\n"
    f"nc: 1\nnames: ['wheat_head']\n")
(NEW / "wheat21_test2.yaml").write_text(
    f"path: {NEW}\ntrain: train_gwhd21.txt\nval: images/test2\nnc: 1\nnames: ['wheat_head']\n")

# safety net: nothing from val / test may appear in the training list
train_names = {p.name for p in train_list}
leak = train_names & (old_val | old_test | {Path(n).stem + ".jpg" for n in test2.name})
assert not leak, f"{len(leak)} validation/test images are in the training list"
print(f"training list: {len(train_list):,} images ({len(train_list) - len(add_train):,} old + {len(add_train):,} new); leaks into val/test: {len(leak)}")
print((NEW / "wheat21.yaml").read_text())

## 11. Training setup

**What it does:** imports YOLO, picks the model size, defines the run name, and gathers the training arguments in **one dictionary**.

**How it works:** every value is *copied from the baseline run* (`runs/detect/baseline/args.yaml`) so the only difference is the data. The reasons behind each value are the same as in `train.ipynb`:

| Argument | Value | Why |
|---|---|---|
| model | `yolo11s.pt` (default) or `yolo11m.pt` | COCO-pretrained. Set `GWD_MODEL=yolo11m` to train the medium model; everything else stays identical, so the two runs differ only in model size |
| `imgsz`, `batch` | 1024, 4 | native image size; the largest batch that fits comfortably in 8 GB |
| `epochs`, `patience` | 60, 20 | same budget as the baseline; stop early if no gain for 20 epochs |
| `mosaic`, `close_mosaic`, `mixup` | 1.0, 10, 0.1 | stitch 4 images for scale/background variety; switch it off for the last 10 epochs |
| `hsv_*` | 0.015 / 0.7 / 0.4 | strong colour jitter for lighting and crop-colour differences between farms |
| `flipud`, `degrees` | 0.5, 15 | overhead photos have no "up" |
| `scale`, `translate` | 0.5, 0.1 | camera distance and framing variation |
| `seed` | 42 | reproducibility |
| optimizer | auto | Ultralytics picks AdamW (lr 0.002) for short runs |

**A helper decides what to do on re-runs.** `training_state()` loads the last checkpoint: while a run is in progress the file contains optimizer state; when it finishes, Ultralytics **strips** that state. So *"has optimizer state"* means the run was interrupted and can resume, and *"no optimizer state"* means it is complete. This makes the notebook safe to run again after a crash or a laptop shutdown.

In [ ]:
import torch
from ultralytics import YOLO

MODEL = os.environ.get("GWD_MODEL", "yolo11s")  # yolo11s or yolo11m
assert MODEL in ("yolo11s", "yolo11m"), MODEL
RUN_NAME = f"{MODEL}_gwhd21"
DATA = str(NEW / "wheat21.yaml")
DATA_TEST2 = str(NEW / "wheat21_test2.yaml")
BASE_WEIGHTS = REPO_ROOT / "weights" / f"{MODEL}.pt"
IMGSZ, BATCH, WORKERS = 1024, 4, 2

train_args = dict(
    data=DATA, epochs=60, imgsz=IMGSZ, batch=BATCH, workers=WORKERS,
    device=0, amp=True, cache=False, patience=20,
    mosaic=1.0, close_mosaic=10, mixup=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, flipud=0.5, degrees=15, scale=0.5, translate=0.1,
    project=str(RUNS_DIR), exist_ok=True, plots=True, seed=42,
)


def training_state(run_name):
    """'new' (no checkpoint), 'interrupted' (can resume) or 'complete'."""
    last = RUNS_DIR / run_name / "weights" / "last.pt"
    if not last.exists():
        return "new"
    ckpt = torch.load(last, map_location="cpu", weights_only=False)
    return "interrupted" if ckpt.get("optimizer") is not None else "complete"


assert BASE_WEIGHTS.exists(), f"missing pretrained weights {BASE_WEIGHTS}"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"model {MODEL} -> run folder runs/detect/{RUN_NAME}")
print("smoke state:", training_state(RUN_NAME + "_smoke"), "| main state:", training_state(RUN_NAME))

## 12. Smoke test: three epochs

**What it does:** trains for just 3 epochs on the full new training set.

**Why:** it costs about 10 minutes and proves that the combined list loads, every label is found, memory fits and nothing crashes, *before* committing to a multi-hour run. A crash at epoch 40 of 60 is far more expensive than one here.

**What to look for in the output:** the line `train: Scanning ... images, N backgrounds, 0 corrupt` should show about **3,935 images** and **0 corrupt**. Validation mAP50-95 after 3 epochs is only a sanity number (expect roughly 0.45 to 0.50). Do not compare it with final scores.

The cell is skipped when the smoke run already finished, or when `RUN_TRAINING` is false.

In [ ]:
if RUN_TRAINING and training_state(RUN_NAME + "_smoke") == "new":
    YOLO(BASE_WEIGHTS).train(**{**train_args, "epochs": 3, "name": RUN_NAME + "_smoke", "plots": False})
    print(f"peak VRAM {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
else:
    print("smoke test skipped (already done or training disabled)")

## 13. The full training run

**What it does:** trains the chosen model for up to 60 epochs on the old + new images. On the RTX 5050, YOLO11s took about **2.4 hours** (about 140 s per epoch with 3,935 images). YOLO11m is roughly twice the compute, so expect about **5 hours**; the smoke test's epoch time gives a better estimate.

**How it works:**
- Each epoch the model sees every training image once. Mosaic stitches four images into one, mixup blends two, and colour/geometry jitter is applied, all randomly.
- After each epoch the model is scored on `val`. The **fitness** (mAP50-95 in this Ultralytics version; older versions blended in 0.1 of mAP50) decides which epoch becomes `best.pt`. Training stops early if fitness does not improve for 20 epochs.
- A checkpoint `last.pt` is written every epoch. If the run is interrupted, running this cell again **resumes** from it (see `training_state`).
- Results go to `runs/detect/yolo11s_gwhd21/`: `results.csv` (one row per epoch), `weights/best.pt`, curves and sample batches.

**What to watch while it runs:**
- *Train loss falling, val mAP rising:* healthy.
- *Val mAP flat from early on:* the new data may be adding noise rather than signal.
- *Loss becomes `NaN`:* learning rate or label problem, so stop and investigate.

In [ ]:
state = training_state(RUN_NAME)
if not RUN_TRAINING:
    print("training disabled (GWD_SKIP_TRAIN=1)")
elif state == "complete":
    print("run already complete - skipping training")
elif state == "interrupted":
    print("resuming interrupted run")
    YOLO(RUNS_DIR / RUN_NAME / "weights" / "last.pt").train(resume=True)
else:
    YOLO(BASE_WEIGHTS).train(**{**train_args, "name": RUN_NAME})
print("state now:", training_state(RUN_NAME))

## 14. Learning curves

**What it does:** plots training loss and validation mAP per epoch from `results.csv`, next to the baseline run for comparison.

**How to read it:**
- **Validation mAP50-95 still rising at the end** means more epochs would help (as the 60-epoch baseline showed).
- **Train loss falling while validation stalls** means overfitting. More data should delay this.
- **Gap between the two runs' validation curves:** on `val` (familiar farms) we expect only a small difference; the real effect, if any, shows up on `test` and `test2` in the next section.

In [ ]:
def load_results(run):
    p = RUNS_DIR / run / "results.csv"
    if not p.exists():
        return None
    d = pd.read_csv(p); d.columns = d.columns.str.strip(); return d


runs = {"YOLO11s, old data": load_results("baseline"),
        "YOLO11s + GWHD 2021": load_results("yolo11s_gwhd21"),
        "YOLO11m + GWHD 2021": load_results("yolo11m_gwhd21")}
runs = {k: v for k, v in runs.items() if v is not None}
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for label, d in runs.items():
    ax[0].plot(d["epoch"], d["train/box_loss"], label=label)
    ax[1].plot(d["epoch"], d["metrics/mAP50-95(B)"], label=label)
ax[0].set_title("training box loss"); ax[1].set_title("validation mAP50-95 (val: familiar farms)")
for a in ax: a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()

## 15. The comparison that matters: every model on val, test and test2

**What it does:** scores every trained model on three evaluation sets, **with exactly the same protocol** (no test-time augmentation, each model at its own training image size) and puts the results side by side.

| Model | Trained on | Image size |
|---|---|---|
| YOLO11s baseline | old data | 1024 |
| YOLO11m combined | old data (+ balanced sampling, extra augmentation) | 1280 |
| YOLO11s + GWHD2021 | **old + new data** | 1024 |
| YOLO11m + GWHD2021 | **old + new data**, bigger model | 1024 |

| Set | Meaning |
|---|---|
| `val` | familiar farms (same as before) |
| `test` | `usask_1` + `ethz_1`, unseen |
| `test2` | official 2021 test farms, unseen and on other continents |

**How it works:** for each model and set, `model.val(...)` runs inference on every image and computes mAP50, mAP50-95, precision and recall against the labels. `split="val"` / `"test"` pick the matching entry in the dataset YAML. A model whose weights are missing is skipped with a message. Evaluating the two earlier models on `test2` is fair because they were trained before this data existed.

**Why re-score the baselines instead of quoting the earlier numbers:** TTA was off for the baseline but on for the combined run in our earlier report. Re-scoring all three under one protocol removes that inconsistency.

In [ ]:
MODELS = {
    "YOLO11s baseline": dict(weights=RUNS_DIR / "baseline" / "weights" / "best.pt", imgsz=1024, batch=4),
    "YOLO11m combined": dict(weights=RUNS_DIR / "yolo11m_combo" / "weights" / "best.pt", imgsz=1280, batch=2),
    "YOLO11s + GWHD2021": dict(weights=RUNS_DIR / "yolo11s_gwhd21" / "weights" / "best.pt", imgsz=1024, batch=4),
    "YOLO11m + GWHD2021": dict(weights=RUNS_DIR / "yolo11m_gwhd21" / "weights" / "best.pt", imgsz=1024, batch=4),
}
EVAL_SETS = {"val": (DATA, "val"), "test": (DATA, "test")}
if HOLDOUT_OFFICIAL_TEST:
    EVAL_SETS["test2"] = (DATA_TEST2, "val")

rows = []
for model_name, cfg in MODELS.items():
    if not cfg["weights"].exists():
        print(f"skipping {model_name}: no weights at {cfg['weights']}")
        continue
    model = YOLO(cfg["weights"])
    for set_name, (yaml_path, split) in EVAL_SETS.items():
        r = model.val(data=yaml_path, split=split, imgsz=cfg["imgsz"], batch=cfg["batch"], augment=False, workers=0,
                      plots=False, verbose=False, project=str(RUNS_DIR / "_eval"), name="tmp", exist_ok=True)
        rows.append(dict(model=model_name, set=set_name, mAP50=r.box.map50, mAP50_95=r.box.map, precision=r.box.mp, recall=r.box.mr))

scores = pd.DataFrame(rows)
table = scores.pivot(index="model", columns="set", values="mAP50_95")[[s for s in EVAL_SETS if s in scores.set.values]].round(4)
table50 = scores.pivot(index="model", columns="set", values="mAP50")[table.columns].round(4)
print("mAP50-95 (tight boxes):"); display(table)
print("mAP50 (finding the heads):"); display(table50)
scores.round(4).to_csv(REPO_ROOT / "runs" / "gwhd2021_scores.csv", index=False)

## 16. Did the new data help? The verdict

**What it does:** computes the change versus the baseline and the generalization gap (`val` minus unseen score) for every model, then prints a plain-language verdict.

**How to read it:**
- **Gap** = how much accuracy is lost on unseen farms. A smaller gap means better generalization.
- **Noise band:** run-to-run variation of about **0.005** mAP is normal, so differences smaller than that are *not* evidence of anything. The verdict only calls something a gain or a loss beyond ±0.005.
- **Two independent unseen sets** (`test` and `test2`) agreeing in direction is much stronger evidence than one.

**When the current model is YOLO11m**, the cell also compares it with YOLO11s trained on the same data. That difference isolates the effect of **model size alone**.

**Decision rules:**
- Both unseen sets improve beyond the noise band: the data was the bottleneck. Next step is to retrain on *all* the data (including `test2`) and look for even more farms.
- Neither improves: label quality or the way unseen farms differ is the limit, so go to per-farm error analysis (next section) before collecting more data.
- Mixed result: inspect which farms win and lose.

In [ ]:
NOISE = 0.005
base = "YOLO11s baseline"
same_data_small = "YOLO11s + GWHD2021"
new = {"yolo11s": "YOLO11s + GWHD2021", "yolo11m": "YOLO11m + GWHD2021"}[MODEL]


def verdict(a, b, title):
    """Print the mAP50-95 change from model b to model a on the unseen sets."""
    unseen = [c for c in ("test", "test2") if c in table.columns]
    print(f"\n{title}  ({a} minus {b})")
    for c in ["val"] + unseen:
        d = table.loc[a, c] - table.loc[b, c]
        word = "IMPROVED" if d > NOISE else "got WORSE" if d < -NOISE else "no change (within noise)"
        print(f"  {c:6s}: {d:+.4f} -> {word}")


if {base, new} <= set(table.index):
    unseen = [c for c in ("test", "test2") if c in table.columns]
    gaps = pd.DataFrame({m: {f"gap val-{c}": round(table.loc[m, "val"] - table.loc[m, c], 4) for c in unseen} for m in table.index}).T
    print("change vs the YOLO11s baseline, every model (mAP50-95):"); display((table - table.loc[base]).round(4))
    print("generalization gap (smaller is better):"); display(gaps)
    verdict(new, base, "effect of the new data" if new == same_data_small else "effect of new data + bigger model")
    if new != same_data_small and same_data_small in table.index:
        verdict(new, same_data_small, "effect of model size alone (same data)")
else:
    print("need both the baseline and the current run to compute the verdict")

## 17. Which farms still fail? Per-farm error analysis

**What it does:** scores the baseline and the models trained on the new data **separately on every farm**, so we can see which farms improved, which did not, and which are simply hard.

**How it works:**
- For each farm, the list of its images is written to a small text file with its own YAML.
- `model.val()` is run once per farm. These are small sets (17 to 750 images) so each run takes seconds.
- Farms with very few images give noisy scores, so the table shows the image count. Treat farms with fewer than about 30 images as indicative only.

**Why this is the most useful analysis:** one average hides the pattern. A model can be excellent on five farms and fail on one, and that one tells you *exactly which kind of data to collect next* (a different crop stage, camera height, or density). It also shows whether the new training farms (for example Japan or Australia) helped the matching test farms.

**Reading the table:** each `delta` column is that model minus the baseline in mAP50-95. Large positive values on a farm similar to a new training farm support the "diverse data helps" hypothesis; a farm where `delta` is negative deserves a look at the predictions.

In [ ]:
old_src = pd.read_csv(REPO_ROOT / "data" / "raw" / "train.csv", usecols=["image_id", "source"]).drop_duplicates("image_id").set_index("image_id")["source"]
farm_images = {}
for p in sorted((OLD / "images" / "test").glob("*.jpg")):
    farm_images.setdefault(f"{old_src[p.stem]} (test)", []).append(p)
if HOLDOUT_OFFICIAL_TEST:
    for r in test2.itertuples():
        farm_images.setdefault(f"{r.domain} (test2)", []).append(NEW / "images" / "test2" / f"{Path(r.name).stem}.jpg")


def write_list_yaml(key, paths):
    d = NEW / "eval_lists"; d.mkdir(exist_ok=True)
    stem = "".join(c if c.isalnum() else "_" for c in key)
    (d / f"{stem}.txt").write_text("\n".join(str(p) for p in paths) + "\n")
    (d / f"{stem}.yaml").write_text(f"path: {d}\ntrain: {stem}.txt\nval: {stem}.txt\nnc: 1\nnames: ['wheat_head']\n")
    return str(d / f"{stem}.yaml")


per_farm = {}
farm_models = [m for m in dict.fromkeys([base, same_data_small, new]) if MODELS[m]["weights"].exists()]
for model_name in farm_models:
    cfg = MODELS[model_name]
    if not cfg["weights"].exists():
        continue
    model = YOLO(cfg["weights"])
    for farm, paths in farm_images.items():
        r = model.val(data=write_list_yaml(farm, paths), split="val", imgsz=cfg["imgsz"], batch=cfg["batch"], augment=False,
                      workers=0, plots=False, verbose=False, project=str(RUNS_DIR / "_eval"), name="tmp", exist_ok=True)
        per_farm.setdefault(farm, {"images": len(paths)})[model_name] = round(r.box.map, 4)

farm_table = pd.DataFrame(per_farm).T
for m in farm_models[1:]:
    farm_table[f"delta {m}"] = (farm_table[m].astype(float) - farm_table[base].astype(float)).round(4)
farm_table.sort_values(f"delta {new}" if f"delta {new}" in farm_table else "images")

## 18. Look at predictions on unseen farms

**What it does:** runs the new model on one image from each of six unseen farms (chosen from the *worst* farms in the table above) and draws the predicted boxes, with the predicted count next to the ground-truth count.

**How it works:** `model.predict(..., conf=0.3)` returns boxes with confidence above 0.3; `result.plot()` draws them. The ground-truth count comes from the label file.

**What to look for:**
- **Predicted count far below the true count:** missed heads (dense or tiny heads, occlusion).
- **Predicted count above the true count:** doubled boxes on one head or false detections on soil and leaves.
- **Boxes present but shifted or loose:** the reason mAP50-95 is much lower than mAP50.

These patterns tell you what the *next* round of data or augmentation should target.

In [ ]:
best_path = MODELS[new]["weights"]
if best_path.exists() and new in farm_table:
    worst = farm_table.sort_values(new).head(6).index.tolist()
    model = YOLO(best_path)
    fig, axes = plt.subplots(2, 3, figsize=(16, 10.5))
    for ax, farm in zip(axes.ravel(), worst):
        p = farm_images[farm][0]
        true_n = len([l for l in (Path(str(p).replace("images", "labels")).with_suffix(".txt")).read_text().split("\n") if l.strip()])
        pred = model.predict(str(p), imgsz=IMGSZ, conf=0.3, verbose=False)[0]
        ax.imshow(pred.plot()[:, :, ::-1])
        ax.set_title(f"{farm}: predicted {len(pred.boxes)} | truth {true_n}", fontsize=10); ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("run the previous sections first (needs the trained model and the per-farm table)")

## 19. Summary and what to do next

After running the notebook, fill in this table from sections 15 and 16 and keep it with your notes.

| Model | `val` mAP50-95 | `test` mAP50-95 | `test2` mAP50-95 |
|---|---:|---:|---:|
| YOLO11s baseline | | | |
| YOLO11m combined | | | |
| **YOLO11s + GWHD2021** | | | |
| **YOLO11m + GWHD2021** | | | |

### How to interpret the outcome

| What you see | What it suggests | Next step |
|---|---|---|
| `test` and `test2` both improve beyond ±0.005 | Training-farm diversity was the bottleneck | Retrain on all new data (`HOLDOUT_OFFICIAL_TEST = False`), then try the larger model and extra augmentation again, since more data may now let them pay off |
| `test2` improves, `test` does not | The new farms resemble `test2` but `ethz_1` / `usask_1` are different again | Per-farm table: what is special about those two? |
| Neither improves | The limit is label noise or the nature of the shift, not farm count | Audit labels on the worst farms; try pseudo-labelling on unlabeled field photos |
| `val` rises but unseen sets do not | Model is fitting the training farms harder | Stronger augmentation or regularization |

### Honest limitations
- **One run per configuration.** Differences under about 0.005 mAP50-95 are within run-to-run noise; repeat with a second seed before trusting a small gain.
- **The new farms carry their own labeling style.** Different groups drew boxes slightly differently, which can cap mAP50-95 even when detection is good.
- **`test2` is a different kind of unseen** than `test` (different continents and growth stages), so the two scores should not be averaged blindly.
- **We changed only the data**, which is why the result can be attributed, but it also means we have not yet combined this with the larger model or higher resolution.

### Files this notebook creates
| Path | Contents |
|---|---|
| `data/gwhd2021/` | The downloaded 10 GB archive (not versioned) |
| `data/yolo21/images`, `labels` | Converted new training images and `test2` |
| `data/yolo21/train_gwhd21.txt` | The exact list of training images |
| `data/yolo21/wheat21.yaml`, `wheat21_test2.yaml` | Dataset descriptions |
| `runs/detect/yolo11s_gwhd21/` | Training results and `best.pt` |
| `runs/gwhd2021_scores.csv` | Final scores for all models and sets |